# 🔍 SEMAINE 7 : Interprétation et Explications des Prédictions

**Projet:** Détection précoce DT1 - Cameroun  
**Objectifs:**
- Comprendre comment le modèle prend ses décisions
- Utiliser SHAP pour l'interprétabilité globale et locale
- Analyser les erreurs de prédiction
- Générer des visualisations pour le mémoire

**Outils:** SHAP (SHapley Additive exPlanations)

**Durée estimée:** 10-12 heures

---

In [ ]:
# Import des bibliothèques
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import joblib
import shap
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print("✅ Bibliothèques importées!")

# Initialiser SHAP JavaScript (pour les visualisations interactives)
shap.initjs()

In [ ]:
# Charger les données
df = pd.read_csv('../2_DONNEES/processed/dataset_clean.csv')
features = ['age', 'IMC', 'glycemie_jeun', 'HbA1c', 'ANP32A_IT1', 'ESCO2', 'NBPF1']
X = df[features]
y = df['diagnostic']

# Division train/test (même seed que précédemment)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"📂 Données chargées: {len(X_train)} train / {len(X_test)} test")

In [ ]:
# Charger le modèle final sauvegardé
try:
    modele = joblib.load('../4_MODELES/models/final_model.pkl')
    print("✅ Modèle final chargé avec succès!")
    print(f"   Type: {type(modele).__name__}")
except FileNotFoundError:
    print("⚠️ Modèle final non trouvé. Entraînement d'un Random Forest de base...")
    from sklearn.ensemble import RandomForestClassifier
    modele = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    modele.fit(X_train, y_train)
    print("✅ Modèle de base entraîné")

## 📊 Partie 1 : Analyse des Prédictions

Avant d'interpréter, analysons les performances du modèle.

In [ ]:
# Prédictions
y_pred = modele.predict(X_test)
y_proba = modele.predict_proba(X_test)[:, 1]

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)

print("📊 PERFORMANCES DU MODÈLE\n")
print(classification_report(y_test, y_pred, target_names=['Sain', 'DT1']))

# Visualisation de la matrice de confusion
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matrice de confusion (nombres)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Sain', 'DT1'], yticklabels=['Sain', 'DT1'],
            ax=axes[0], cbar_kws={'label': 'Nombre de patients'})
axes[0].set_xlabel('Diagnostic prédit', fontsize=12)
axes[0].set_ylabel('Diagnostic réel', fontsize=12)
axes[0].set_title('Matrice de Confusion', fontsize=14, fontweight='bold')

# Matrice de confusion (pourcentages)
cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Greens',
            xticklabels=['Sain', 'DT1'], yticklabels=['Sain', 'DT1'],
            ax=axes[1], cbar_kws={'label': 'Pourcentage (%)'})
axes[1].set_xlabel('Diagnostic prédit', fontsize=12)
axes[1].set_ylabel('Diagnostic réel', fontsize=12)
axes[1].set_title('Matrice de Confusion (%)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# Identifier les erreurs
faux_positifs = np.where((y_test == 0) & (y_pred == 1))[0]
faux_negatifs = np.where((y_test == 1) & (y_pred == 0))[0]

print(f"\n❌ Erreurs de prédiction:")
print(f"   Faux Positifs (Sain prédit DT1): {len(faux_positifs)} cas")
print(f"   Faux Négatifs (DT1 prédit Sain): {len(faux_negatifs)} cas ⚠️ (CRITIQUE)")

## 🔍 Partie 2 : SHAP - Importance Globale des Features

SHAP (SHapley Additive exPlanations) = méthode basée sur la théorie des jeux pour expliquer les prédictions.

In [ ]:
print("🔍 Calcul des valeurs SHAP...")
print("⏳ Cela peut prendre 1-3 minutes...\n")

# Créer l'explainer SHAP (TreeExplainer pour RF/XGBoost)
explainer = shap.TreeExplainer(modele)

# Calculer les valeurs SHAP sur un échantillon du test set (plus rapide)
X_test_sample = X_test.sample(min(200, len(X_test)), random_state=42)
shap_values = explainer.shap_values(X_test_sample)

# Si le modèle retourne une liste (pour chaque classe)
if isinstance(shap_values, list):
    shap_values_dt1 = shap_values[1]  # Valeurs SHAP pour la classe DT1
else:
    shap_values_dt1 = shap_values

print("✅ Valeurs SHAP calculées!")
print(f"   Forme: {shap_values_dt1.shape} (patients × features)")

In [ ]:
# SHAP Summary Plot (importance globale)
print("📊 SHAP Summary Plot (importance des features)\n")

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_dt1, X_test_sample, plot_type="bar", show=False)
plt.title('Importance Globale des Features (SHAP)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 Interprétation:")
print("   - Les features en haut ont le plus d'impact sur les prédictions")
print("   - L'importance SHAP est basée sur la contribution moyenne absolue")

In [ ]:
# SHAP Beeswarm Plot (impact et distribution)
print("📊 SHAP Beeswarm Plot (impact des valeurs)\n")

plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values_dt1, X_test_sample, show=False)
plt.title('Impact des Features sur les Prédictions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 Interprétation:")
print("   - Axe X: Impact SHAP (positif = augmente prob. DT1, négatif = diminue)")
print("   - Couleur: Valeur de la feature (rouge = élevée, bleu = faible)")
print("   - Exemple: Si glycémie élevée (rouge) est à droite → augmente prob. DT1")

## 🎯 Partie 3 : SHAP - Explication Individuelle

Expliquer la prédiction pour un patient spécifique.

In [ ]:
# Sélectionner un cas intéressant: un patient DT1 correctement détecté
dt1_correct = X_test[(y_test == 1) & (y_pred == 1)].iloc[0:1]
idx_patient = dt1_correct.index[0]

print(f"🔬 EXPLICATION POUR LE PATIENT {idx_patient}\n")
print("Caractéristiques du patient:")
print(dt1_correct.T)

# Prédiction
proba_dt1_patient = modele.predict_proba(dt1_correct)[0, 1]
print(f"\nPrédiction: DT1 avec probabilité {proba_dt1_patient*100:.1f}%")

# Calculer SHAP pour ce patient
shap_values_patient = explainer.shap_values(dt1_correct)
if isinstance(shap_values_patient, list):
    shap_values_patient = shap_values_patient[1]

# Force plot
print("\n📊 SHAP Force Plot (explication visuelle)")
shap.force_plot(
    explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value,
    shap_values_patient[0],
    dt1_correct.iloc[0],
    matplotlib=True,
    show=False
)
plt.title(f'Explication de la prédiction pour le patient {idx_patient}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Interprétation du Force Plot:")
print("   - Base value (gris): Prédiction moyenne du modèle")
print("   - Flèches rouges: Features qui augmentent la probabilité de DT1")
print("   - Flèches bleues: Features qui diminuent la probabilité de DT1")
print("   - Output value: Prédiction finale pour ce patient")

In [ ]:
# Waterfall plot (alternative plus lisible)
print("📊 SHAP Waterfall Plot\n")

shap.waterfall_plot(
    shap.Explanation(
        values=shap_values_patient[0],
        base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value,
        data=dt1_correct.iloc[0].values,
        feature_names=features
    ),
    show=False
)
plt.title(f'Cascade de contributions - Patient {idx_patient}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 Lecture du Waterfall:")
print("   - Chaque barre montre la contribution d'une feature")
print("   - Part de E[f(x)] (base) et arrive à f(x) (prédiction finale)")

## ❌ Partie 4 : Analyse des Erreurs

Analyser les cas où le modèle se trompe pour identifier ses limites.

In [ ]:
print("❌ ANALYSE DES FAUX NÉGATIFS (DT1 manqués)\n")

if len(faux_negatifs) > 0:
    # Sélectionner les faux négatifs
    X_fn = X_test.iloc[faux_negatifs]
    y_proba_fn = y_proba[faux_negatifs]
    
    print(f"Nombre de faux négatifs: {len(faux_negatifs)}")
    print(f"Probabilités DT1 moyennes: {y_proba_fn.mean()*100:.1f}% (vs seuil 50%)\n")
    
    # Comparer avec les vrais positifs
    vrais_positifs = np.where((y_test == 1) & (y_pred == 1))[0]
    X_vp = X_test.iloc[vrais_positifs]
    
    # Statistiques comparatives
    comparaison = pd.DataFrame({
        'Feature': features,
        'Moyenne FN': X_fn.mean(),
        'Moyenne VP': X_vp.mean(),
        'Différence': X_fn.mean() - X_vp.mean()
    }).sort_values('Différence', key=abs, ascending=False)
    
    print("📊 Différences entre Faux Négatifs et Vrais Positifs:")
    print(comparaison.round(2))
    
    # Visualisation
    fig, ax = plt.subplots(figsize=(10, 6))
    x = np.arange(len(features))
    width = 0.35
    
    ax.bar(x - width/2, X_fn.mean(), width, label='Faux Négatifs', color='red', alpha=0.7)
    ax.bar(x + width/2, X_vp.mean(), width, label='Vrais Positifs', color='green', alpha=0.7)
    
    ax.set_xlabel('Feature', fontsize=12)
    ax.set_ylabel('Valeur moyenne', fontsize=12)
    ax.set_title('Profil des Faux Négatifs vs Vrais Positifs', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(features, rotation=45, ha='right')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("\n💡 Interprétation:")
    print("   - Les faux négatifs ont souvent des valeurs 'limites' (proches des sains)")
    print("   - Identifier les features les plus différentes aide à comprendre les erreurs")
else:
    print("✅ Aucun faux négatif! Modèle parfait sur cette métrique.")

In [ ]:
print("❌ ANALYSE DES FAUX POSITIFS (Sains classés DT1)\n")

if len(faux_positifs) > 0:
    X_fp = X_test.iloc[faux_positifs]
    y_proba_fp = y_proba[faux_positifs]
    
    print(f"Nombre de faux positifs: {len(faux_positifs)}")
    print(f"Probabilités DT1 moyennes: {y_proba_fp.mean()*100:.1f}% (> seuil 50%)\n")
    
    # Cas le plus confiant (mais faux)
    idx_plus_confiant = faux_positifs[np.argmax(y_proba_fp)]
    proba_plus_confiant = y_proba[idx_plus_confiant]
    
    print(f"📌 Faux positif le plus confiant:")
    print(f"   Patient index: {X_test.iloc[idx_plus_confiant].name}")
    print(f"   Probabilité DT1: {proba_plus_confiant*100:.1f}%")
    print(f"   Caractéristiques:")
    print(X_test.iloc[idx_plus_confiant])
    
    print("\n💡 Ce patient sain ressemble beaucoup à un DT1 selon les features mesurées")
else:
    print("✅ Aucun faux positif!")

## 📈 Partie 5 : SHAP Dependence Plots

Analyser l'impact d'une feature spécifique sur les prédictions.

In [ ]:
# Dependence plot pour les 3 features les plus importantes
top_features = ['glycemie_jeun', 'HbA1c', 'ESCO2']  # Adapter selon vos résultats

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, feature in enumerate(top_features):
    shap.dependence_plot(
        feature, 
        shap_values_dt1, 
        X_test_sample,
        ax=axes[i],
        show=False
    )
    axes[i].set_title(f'Impact de {feature}', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("💡 Interprétation des Dependence Plots:")
print("   - Axe X: Valeur de la feature")
print("   - Axe Y: Impact SHAP (contribution à la prédiction)")
print("   - Couleur: Valeur d'une autre feature (interaction)")
print("   - Tendance croissante = valeurs élevées augmentent prob. DT1")

## 💾 Partie 6 : Sauvegarde des Visualisations pour le Mémoire

In [ ]:
import os
os.makedirs('../6_MEMOIRE_LATEX/figures', exist_ok=True)

print("💾 Sauvegarde des figures pour le mémoire LaTeX...\n")

# Figure 1: Matrice de confusion
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Sain', 'DT1'], yticklabels=['Sain', 'DT1'],
            ax=ax, cbar_kws={'label': 'Nombre de patients'})
ax.set_xlabel('Diagnostic prédit', fontsize=12)
ax.set_ylabel('Diagnostic réel', fontsize=12)
ax.set_title('Matrice de Confusion - Modèle Final', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../6_MEMOIRE_LATEX/figures/matrice_confusion.png', dpi=300, bbox_inches='tight')
plt.close()
print("✅ matrice_confusion.png")

# Figure 2: SHAP Summary (bar)
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_dt1, X_test_sample, plot_type="bar", show=False)
plt.title('Importance des Features (SHAP)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../6_MEMOIRE_LATEX/figures/shap_importance.png', dpi=300, bbox_inches='tight')
plt.close()
print("✅ shap_importance.png")

# Figure 3: SHAP Beeswarm
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values_dt1, X_test_sample, show=False)
plt.title('Impact des Features (SHAP)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../6_MEMOIRE_LATEX/figures/shap_beeswarm.png', dpi=300, bbox_inches='tight')
plt.close()
print("✅ shap_beeswarm.png")

print("\n💾 Toutes les figures sauvegardées dans ../6_MEMOIRE_LATEX/figures/")
print("📝 À intégrer dans le mémoire avec \\includegraphics{}")

## 🎯 Conclusion

### Points clés de l'interprétabilité:

1. **SHAP** permet de comprendre **comment** et **pourquoi** le modèle prédit
2. **Importance globale** montre les features les plus influentes en moyenne
3. **Explications locales** permettent de justifier chaque prédiction individuelle
4. **Analyse des erreurs** identifie les limites du modèle
5. **Dependence plots** révèlent les relations non-linéaires

### Applications cliniques:

✅ **Confiance des médecins:** Comprendre les décisions du modèle  
✅ **Validation biologique:** Vérifier que le modèle utilise des features pertinentes  
✅ **Détection d'anomalies:** Identifier les cas atypiques  
✅ **Communication:** Expliquer les résultats aux patients et familles  

### Pour le mémoire:

📊 **Figures générées:** matrice_confusion.png, shap_importance.png, shap_beeswarm.png  
📝 **À inclure:** Section "Interprétabilité" dans Résultats + Discussion  
🔬 **À discuter:** Validation biologique des features importantes (ESCO2, HbA1c, etc.)  

---

## 🎉 Félicitations!

Vous avez terminé les 7 semaines de notebooks pédagogiques! Vous maîtrisez maintenant:

✅ Manipulation de données (NumPy, Pandas)  
✅ Visualisation (Matplotlib, Seaborn)  
✅ Modélisation ML (RF, XGBoost, LightGBM)  
✅ Optimisation (GridSearch, validation croisée)  
✅ Interprétabilité (SHAP)  
✅ Analyse critique des résultats  

### 📅 Prochaines étapes:

1. **Rédaction du mémoire** (sections Résultats, Discussion)
2. **Finalisation de l'application** Streamlit avec le modèle final
3. **Préparation de la soutenance** (slides + démo)

---

*Notebook créé pour le projet DT1 Cameroun - Master 2 Biophysique - 2025*